In [1]:
# Cell 1: Environment Check & Install
import subprocess
subprocess.run(["nvidia-smi"], check=True)

!pip install ultralytics==8.2.0 -q


import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


Sat Feb 28 14:14:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2: Imports + PyTorch 2.6 Compatibility Patch

import os, sys, time, shutil, glob, math, yaml, csv
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from pathlib import Path
from tqdm.notebook import tqdm
from ultralytics import YOLO

# ── PyTorch 2.6 fix ──────────────────────────────────────────────────────────
# PyTorch 2.6 changed torch.load default to weights_only=True.
# Ultralytics .pt files contain DetectionModel objects which are blocked by default.
# We explicitly allowlist them here — this is safe because yolov8n.pt is from
# the official Ultralytics GitHub release.
import torch.serialization
from ultralytics.nn.tasks import DetectionModel
from ultralytics.nn.modules import (
    Conv, C2f, SPPF, Detect, DFL
)
torch.serialization.add_safe_globals([
    DetectionModel,
    Conv, C2f, SPPF, Detect, DFL,
])
print("PyTorch 2.6 safe_globals patch applied ✓")
# ─────────────────────────────────────────────────────────────────────────────

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
print("PyTorch version:", torch.__version__)


PyTorch 2.6 safe_globals patch applied ✓
Using device: cuda
PyTorch version: 2.9.0+cu126


In [3]:
# Cell 3: Dataset Paths (CORRECTED)
BASE = Path("/kaggle/input/datasets/gaweshgomes/llvip-rgb-thermal-yolo-format/processed")

RGB_TRAIN   = BASE / "rgb/images/train"
RGB_VAL     = BASE / "rgb/images/val"
THERM_TRAIN = BASE / "thermal/images/train"
THERM_VAL   = BASE / "thermal/images/val"
LBL_TRAIN   = BASE / "rgb/labels/train"
LBL_VAL     = BASE / "rgb/labels/val"

WORK     = Path("/kaggle/working/adaptive_fusion_v2")
CKPT_DIR = WORK / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Sanity check
rgb_train   = sorted(list(RGB_TRAIN.glob("*.jpg"))   + list(RGB_TRAIN.glob("*.png")))
therm_train = sorted(list(THERM_TRAIN.glob("*.jpg")) + list(THERM_TRAIN.glob("*.png")))
rgb_val     = sorted(list(RGB_VAL.glob("*.jpg"))     + list(RGB_VAL.glob("*.png")))
therm_val   = sorted(list(THERM_VAL.glob("*.jpg"))   + list(THERM_VAL.glob("*.png")))

print(f"RGB    train: {len(rgb_train):,}   val: {len(rgb_val):,}")
print(f"Thermal train: {len(therm_train):,}   val: {len(therm_val):,}")

assert len(rgb_train) == len(therm_train), "Mismatch in train pairs!"
assert len(rgb_val)   == len(therm_val),   "Mismatch in val pairs!"
print(" All pairs matched")


RGB    train: 12,025   val: 3,463
Thermal train: 12,025   val: 3,463
 All pairs matched


In [4]:
# Verify first and last filename pairs match
print("First pair:")
print(f"  RGB   : {rgb_train[0].name}")
print(f"  Therm : {therm_train[0].name}")

print("Last pair:")
print(f"  RGB   : {rgb_train[-1].name}")
print(f"  Therm : {therm_train[-1].name}")

# Check a few are in the same order
mismatches = [(r.name, t.name) for r, t in zip(rgb_train[:10], therm_train[:10]) if r.name != t.name]
if mismatches:
    print(f"  Filename mismatches found: {mismatches}")
else:
    print(" All filename pairs aligned")


First pair:
  RGB   : 010001.jpg
  Therm : 010001.jpg
Last pair:
  RGB   : 250423.jpg
  Therm : 250423.jpg
 All filename pairs aligned


In [5]:
# Cell 4: Dual-Stream Dataset
# Returns RGB and Thermal as SEPARATE tensors — never blended, never stacked.
# This is what enables genuine separate-stream feature extraction.

class DualStreamDataset(Dataset):
    def __init__(self, rgb_dir, therm_dir, lbl_dir, img_size=640):
        self.img_size  = img_size
        self.rgb_files = sorted(
            list(Path(rgb_dir).glob("*.jpg")) + list(Path(rgb_dir).glob("*.png"))
        )
        self.therm_dir = Path(therm_dir)
        self.lbl_dir   = Path(lbl_dir)

    def __len__(self):
        return len(self.rgb_files)

    def _load_image(self, path):
        img = cv2.imread(str(path))           # BGR uint8 [H,W,3]
        if img is None:
            raise FileNotFoundError(f"Cannot read: {path}")
        if img.shape[:2] != (self.img_size, self.img_size):
            img = cv2.resize(img, (self.img_size, self.img_size))
        img = img.astype(np.float32) / 255.0  # [0,1]
        return torch.from_numpy(img).permute(2, 0, 1)  # [3, H, W]

    def _load_labels(self, stem):
        lbl_path = self.lbl_dir / (stem + ".txt")
        labels = []
        if lbl_path.exists() and lbl_path.stat().st_size > 0:
            with open(lbl_path) as f:
                for line in f:
                    vals = list(map(float, line.strip().split()))
                    if len(vals) == 5:
                        labels.append(vals)
        if labels:
            return torch.tensor(labels, dtype=torch.float32)  # [N, 5]
        return torch.zeros((0, 5), dtype=torch.float32)

    def __getitem__(self, idx):
        rgb_path   = self.rgb_files[idx]
        stem       = rgb_path.stem
        therm_path = self.therm_dir / rgb_path.name

        rgb   = self._load_image(rgb_path)
        therm = self._load_image(therm_path)

        # Illumination = mean brightness of RGB image (not thermal)
        # This is the per-image adaptive signal
        illumination = rgb.mean().item()

        labels = self._load_labels(stem)
        return rgb, therm, labels, illumination, stem


def collate_fn(batch):
    rgbs, therms, all_labels, illums, stems = zip(*batch)
    rgb_batch   = torch.stack(rgbs,   0)   # [B, 3, H, W]
    therm_batch = torch.stack(therms, 0)   # [B, 3, H, W]
    illum_batch = torch.tensor(illums, dtype=torch.float32)  # [B]

    # Prepend batch index to each label row → [batch_idx, cls, cx, cy, w, h]
    batched_labels = []
    for i, lbl in enumerate(all_labels):
        if lbl.shape[0] > 0:
            bi = torch.full((lbl.shape[0], 1), float(i))
            batched_labels.append(torch.cat([bi, lbl], dim=1))
    batched_labels = torch.cat(batched_labels, 0) if batched_labels \
                     else torch.zeros((0, 6), dtype=torch.float32)

    return rgb_batch, therm_batch, batched_labels, illum_batch, list(stems)


# Smoke test
ds = DualStreamDataset(RGB_TRAIN, THERM_TRAIN, LBL_TRAIN)
rgb, therm, lbl, illum, stem = ds[0]
print(f"RGB tensor   : {rgb.shape}   range [{rgb.min():.3f}, {rgb.max():.3f}]")
print(f"Thermal tensor: {therm.shape}  range [{therm.min():.3f}, {therm.max():.3f}]")
print(f"Labels        : {lbl.shape}")
print(f"Illumination  : {illum:.4f}  (expected ~0.10–0.25 for LLVIP nighttime)")
print(f"Dataset size  : {len(ds):,}")


RGB tensor   : torch.Size([3, 640, 640])   range [0.000, 0.914]
Thermal tensor: torch.Size([3, 640, 640])  range [0.000, 0.780]
Labels        : torch.Size([6, 5])
Illumination  : 0.1226  (expected ~0.10–0.25 for LLVIP nighttime)
Dataset size  : 12,025


In [6]:
# Cell 5: Attention Modules
# These operate on 256-channel FEATURE MAPS — not pixels.
# F_rgb and F_therm are genuinely separate at this point (separate backbone passes).

class ChannelAttention(nn.Module):
    """
    Squeeze-and-Excitation on RGB feature map.
    Asks: "Which of the 256 feature channels carry useful RGB signal?"
    In nighttime scenes, many RGB channels carry mostly noise.
    This module learns to suppress them and amplify the useful ones.
    Uses both avg-pool and max-pool for richer channel statistics.
    """
    def __init__(self, channels=256, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1 = nn.Linear(channels, hidden)
        self.fc2 = nn.Linear(hidden, channels)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: [B, C, H, W]
        avg = self.fc2(F.relu(self.fc1(self.avg_pool(x).flatten(1))))
        mx  = self.fc2(F.relu(self.fc1(self.max_pool(x).flatten(1))))
        scale = self.sigmoid(avg + mx)            # [B, C]
        return x * scale.unsqueeze(-1).unsqueeze(-1)  # [B, C, H, W]


class SpatialAttention(nn.Module):
    """
    Spatial gating on Thermal feature map.
    Asks: "WHERE in this 40×40 grid are the hot (pedestrian) regions?"
    Outputs a per-location weight map — high where bodies are, low elsewhere.
    """
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size,
                                 padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: [B, C, H, W]
        avg_map = x.mean(dim=1, keepdim=True)       # [B, 1, H, W]
        max_map = x.max(dim=1, keepdim=True)[0]     # [B, 1, H, W]
        scale   = self.sigmoid(
            self.conv(torch.cat([avg_map, max_map], dim=1))
        )                                            # [B, 1, H, W]
        return x * scale                             # [B, C, H, W]


class IlluminationAdaptiveFusion(nn.Module):
    """
    The core fusion module. Takes SEPARATE RGB and Thermal feature maps.

    Three-layer adaptivity:
      1. ChannelAttention(F_rgb)  → which channels survive in dark RGB
      2. SpatialAttention(F_therm)→ where thermal bodies are spatially
      3. Illumination scalar      → global RGB vs Thermal blend ratio

    Learnable temp and bias mean the network adjusts its own
    fusion sensitivity based on what the detection loss rewards.
    """
    def __init__(self, channels=256):
        super().__init__()
        self.rgb_ca   = ChannelAttention(channels=channels, reduction=16)
        self.therm_sa = SpatialAttention(kernel_size=7)
        # Learnable sigmoid parameters — start at sigmoid(0.17*4 - 1.0) ≈ 0.44
        self.temp = nn.Parameter(torch.tensor(4.0))
        self.bias = nn.Parameter(torch.tensor(-1.0))

    def forward(self, F_rgb, F_therm, illumination):
        """
        F_rgb        : [B, 256, 40, 40] — backbone output for RGB input
        F_therm      : [B, 256, 40, 40] — backbone output for Thermal input
        illumination : [B]               — mean RGB brightness per image

        These feature maps are NEVER mixed before this point.
        The backbone ran separately on each modality.
        """
        # Attend each stream independently
        F_rgb_att   = self.rgb_ca(F_rgb)     # [B, 256, 40, 40]
        F_therm_att = self.therm_sa(F_therm) # [B, 256, 40, 40]

        # Per-image illumination weight
        # Dark image (illum≈0.17): rgb_w≈0.44, more thermal used
        # Bright image (illum≈0.70): rgb_w≈0.87, more RGB used
        rgb_w   = torch.sigmoid(
            illumination * self.temp + self.bias
        ).view(-1, 1, 1, 1)                  # [B, 1, 1, 1]
        therm_w = 1.0 - rgb_w

        fused = rgb_w * F_rgb_att + therm_w * F_therm_att  # [B, 256, 40, 40]
        return fused


# Quick test
ca   = ChannelAttention(256).to(DEVICE)
sa   = SpatialAttention(7).to(DEVICE)
fuse = IlluminationAdaptiveFusion(256).to(DEVICE)

dummy_rgb   = torch.randn(2, 256, 40, 40).to(DEVICE)
dummy_therm = torch.randn(2, 256, 40, 40).to(DEVICE)
dummy_illum = torch.tensor([0.17, 0.19]).to(DEVICE)

out = fuse(dummy_rgb, dummy_therm, dummy_illum)
print(f"Fusion output shape: {out.shape}   expected [2, 256, 40, 40]")
print(f"Fusion parameters  : {sum(p.numel() for p in fuse.parameters()):,}")


Fusion output shape: torch.Size([2, 256, 40, 40])   expected [2, 256, 40, 40]
Fusion parameters  : 8,564


In [7]:
# Cell 5b: Patch Ultralytics torch_safe_load for PyTorch 2.6
# 
# PyTorch 2.6 changed torch.load default to weights_only=True.
# Ultralytics' torch_safe_load doesn't pass weights_only=False explicitly.
# We monkey-patch it to force weights_only=False.
# This is safe — yolov8n.pt is from the official Ultralytics GitHub release.

import ultralytics.nn.tasks as _tasks

def _patched_torch_safe_load(weight):
    from ultralytics.utils.downloads import attempt_download_asset
    from pathlib import Path
    file = attempt_download_asset(weight)
    try:
        ckpt = torch.load(file, map_location="cpu", weights_only=False)
        return ckpt, file
    except Exception as e:
        raise e

_tasks.torch_safe_load = _patched_torch_safe_load

# Also patch the reference used inside attempt_load_one_weight
import ultralytics.nn.tasks
ultralytics.nn.tasks.torch_safe_load = _patched_torch_safe_load

print(" torch_safe_load patched for PyTorch 2.6 compatibility")
print("   weights_only=False applied — safe for official Ultralytics weights")


 torch_safe_load patched for PyTorch 2.6 compatibility
   weights_only=False applied — safe for official Ultralytics weights


In [8]:
# Cell 6: AdaptiveFusionYOLO — Corrected Multi-Scale Forward Pass

class AdaptiveFusionYOLO(nn.Module):
    """
    Dual-stream YOLOv8n with shared backbone.
    
    RGB and Thermal pass through backbone SEPARATELY.
    Fusion happens at Layer 8 (256ch, 40×40) — the deepest semantic features.
    Layers 4 and 6 skip connections come from the Thermal stream
    (thermal is the stronger modality on nighttime LLVIP data).
    The fused Layer 8 + thermal skip connections feed the neck correctly.
    """

    def __init__(self, weights="yolov8n.pt"):
        super().__init__()

        base       = YOLO(weights)
        full_model = base.model.to(DEVICE)

        # ── Store individual layers for explicit forward control ──────────────
        self.layers = full_model.model  # nn.Sequential of all 23 layers
        self.fusion = IlluminationAdaptiveFusion(channels=256).to(DEVICE)
        self.full_model = full_model

        # YOLOv8n layer roles:
        # 0       Conv(3→16)       stride 2   → [B, 16,  320, 320]
        # 1       Conv(16→32)      stride 2   → [B, 32,  160, 160]
        # 2       C2f(32→32)                  → [B, 32,  160, 160]
        # 3       Conv(32→64)      stride 2   → [B, 64,   80,  80]
        # 4       C2f(64→64)                  → [B, 64,   80,  80]  ← skip P3
        # 5       Conv(64→128)     stride 2   → [B, 128,  40,  40]
        # 6       C2f(128→128)                → [B, 128,  40,  40]  ← skip P4
        # 7       Conv(128→256)    stride 2   → [B, 256,  20,  20]
        # 8       C2f(256→256)                → [B, 256,  20,  20]  ← FUSE HERE
        # 9       SPPF(256→256)               → [B, 256,  20,  20]
        # 10–22   Neck + Detect head (uses skip connections from 4 and 6)

        print("AdaptiveFusionYOLO ready:")
        print(f"  Total layers    : {len(self.layers)}")
        print(f"  Fusion at       : Layer 8 (C2f 256ch, 20×20)")
        print(f"  Skip connections: Layers 4 (64ch) and 6 (128ch) from Thermal stream")
        print(f"  Fusion params   : {sum(p.numel() for p in self.fusion.parameters()):,}")
        total = sum(p.numel() for p in self.parameters())
        print(f"  Total params    : {total:,}")

    def _run_backbone(self, x):
        """
        Run layers 0–8 and return intermediate outputs needed for neck.
        Returns: (p3, p4, p5)
          p3: Layer 4 output  [B,  64, 80, 80]
          p4: Layer 6 output  [B, 128, 40, 40]
          p5: Layer 8 output  [B, 256, 20, 20]
        """
        out = x
        for i, layer in enumerate(self.layers[:9]):
            out = layer(out)
            if i == 4:
                p3 = out   # [B, 64,  80, 80]
            if i == 6:
                p4 = out   # [B, 128, 40, 40]
        p5 = out           # [B, 256, 20, 20]  (after layer 8)
        return p3, p4, p5

    def forward(self, rgb, therm, illumination):
        """
        rgb         : [B, 3, H, W]
        therm       : [B, 3, H, W]
        illumination: [B]
        """
        # ── Two separate backbone passes ──────────────────────────────────────
        rgb_p3,   rgb_p4,   rgb_p5   = self._run_backbone(rgb)
        therm_p3, therm_p4, therm_p5 = self._run_backbone(therm)

        # ── Fuse at Layer 8 (deepest semantic level) ──────────────────────────
        illum = illumination.to(DEVICE)
        fused_p5 = self.fusion(rgb_p5, therm_p5, illum)  # [B, 256, 20, 20]

        # ── Skip connections: use Thermal stream (stronger at night) ──────────
        # These feed the neck's Concat/Upsample operations
        skip_p3 = therm_p3   # [B,  64, 80, 80]
        skip_p4 = therm_p4   # [B, 128, 40, 40]

        # ── SPPF + Neck + Head ────────────────────────────────────────────────
        # YOLOv8 neck expects the model's internal save list.
        # We run layers 9 onwards manually, feeding the correct tensors
        # at each Concat point.
        #
        # The neck's save indices in YOLOv8n are: [4, 6, 9, 12, 15, 18, 21]
        # We intercept at the right points using the layer's 'f' attribute
        # which tells us which previous layer outputs it needs.

        y = {}          # store outputs by layer index
        y[4]  = skip_p3
        y[6]  = skip_p4
        y[8]  = fused_p5

        out = fused_p5
        for i, layer in enumerate(self.layers[9:], start=9):
            # Check if this layer needs inputs from saved layers (Concat etc.)
            f = layer.f  # from-index: which layer(s) feed this one
            if isinstance(f, int):
                if f == -1:
                    inp = out
                else:
                    inp = y[f]
                out = layer(inp)
            else:
                # Multi-input layer (Concat) — collect from saved outputs
                inputs = []
                for fi in f:
                    if fi == -1:
                        inputs.append(out)
                    else:
                        inputs.append(y[fi])
                out = layer(inputs)

            y[i] = out

        return out


# Instantiate
model = AdaptiveFusionYOLO("yolov8n.pt")
model = model.to(DEVICE)

# VRAM check
dummy_rgb   = torch.randn(8, 3, 640, 640).to(DEVICE)
dummy_therm = torch.randn(8, 3, 640, 640).to(DEVICE)
dummy_illum = torch.full((8,), 0.17).to(DEVICE)

with torch.no_grad():
    out = model(dummy_rgb, dummy_therm, dummy_illum)

# out is a list of detection tensors from the three Detect head scales
if isinstance(out, (list, tuple)):
    print(f"\nForward pass output ({len(out)} tensors):")
    for i, o in enumerate(out):
        if hasattr(o, 'shape'):
            print(f"  [{i}] shape: {o.shape}")
        elif isinstance(o, (list, tuple)):
            print(f"  [{i}] nested: {[x.shape for x in o if hasattr(x,'shape')]}")
else:
    print(f"\nForward pass output: {out.shape}")

allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
print(f"\nVRAM with batch=8:")
print(f"  Allocated: {allocated:.2f} GB")
print(f"  Reserved : {reserved:.2f} GB")
print(f"  T4 limit : 15.0 GB")
print(f"  Status   : {' Safe' if reserved < 12 else '⚠️ Tight — switch to batch=4'}")

del dummy_rgb, dummy_therm, dummy_illum
torch.cuda.empty_cache()


100%|██████████| 6.23M/6.23M [00:00<00:00, 146MB/s]


AdaptiveFusionYOLO ready:
  Total layers    : 23
  Fusion at       : Layer 8 (C2f 256ch, 20×20)
  Skip connections: Layers 4 (64ch) and 6 (128ch) from Thermal stream
  Fusion params   : 8,564
  Total params    : 3,165,764

Forward pass output (2 tensors):
  [0] shape: torch.Size([8, 84, 8400])
  [1] nested: [torch.Size([8, 144, 80, 80]), torch.Size([8, 144, 40, 40]), torch.Size([8, 144, 20, 20])]

VRAM with batch=8:
  Allocated: 0.19 GB
  Reserved : 0.57 GB
  T4 limit : 15.0 GB
  Status   :  Safe


In [9]:
# Cell 7: DataLoaders
BATCH_SIZE  = 8    # Start at 8 for VRAM safety. Try 16 if Cell 6 showed <10GB.
NUM_WORKERS = 4
IMG_SIZE    = 640

train_ds = DualStreamDataset(RGB_TRAIN, THERM_TRAIN, LBL_TRAIN, IMG_SIZE)
val_ds   = DualStreamDataset(RGB_VAL,   THERM_VAL,   LBL_VAL,   IMG_SIZE)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=collate_fn,
    pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=collate_fn,
    pin_memory=True
)

print(f"Train: {len(train_ds):,} images → {len(train_loader):,} batches of {BATCH_SIZE}")
print(f"Val  : {len(val_ds):,} images → {len(val_loader):,} batches of {BATCH_SIZE}")


Train: 12,025 images → 1,503 batches of 8
Val  : 3,463 images → 433 batches of 8


In [10]:
# Cell 8: Generate Per-Image Adaptive Fused Images
# Each image gets its OWN rgb_w based on its own brightness
# This is the core of adaptive fusion — different blend per image

WORK = Path("/kaggle/working/adaptive_fusion_v2")

FUSED_TRAIN_DIR = WORK / "fused_images" / "images" / "train"
FUSED_VAL_DIR   = WORK / "fused_images" / "images" / "val"
FUSED_LBL_TRAIN = WORK / "fused_images" / "labels" / "train"
FUSED_LBL_VAL   = WORK / "fused_images" / "labels" / "val"

for p in [FUSED_TRAIN_DIR, FUSED_VAL_DIR, FUSED_LBL_TRAIN, FUSED_LBL_VAL]:
    p.mkdir(parents=True, exist_ok=True)

def generate_fused(rgb_files, therm_dir, lbl_src_dir,
                   out_img_dir, out_lbl_dir, split):
    skipped = 0
    rgb_w_log = []

    for rgb_path in tqdm(rgb_files, desc=f"Fusing {split}"):
        stem       = rgb_path.stem
        therm_path = Path(therm_dir) / rgb_path.name
        if not therm_path.exists():
            skipped += 1
            continue

        rgb   = cv2.imread(str(rgb_path))
        therm = cv2.imread(str(therm_path))
        if rgb is None or therm is None:
            skipped += 1
            continue

        if rgb.shape[:2] != (640, 640):
            rgb   = cv2.resize(rgb,   (640, 640))
        if therm.shape[:2] != (640, 640):
            therm = cv2.resize(therm, (640, 640))

        rgb_f   = rgb.astype(np.float32)   / 255.0
        therm_f = therm.astype(np.float32) / 255.0

        # ── TRUE PER-IMAGE ADAPTIVE WEIGHT ───────────────────────────────────
        illum = rgb_f.mean()                               # this image's brightness
        rgb_w = 1.0 / (1.0 + np.exp(-(illum * 4.0 - 1.0))) # sigmoid — unique per image
        # Dark image  → rgb_w ≈ 0.40 → 60% thermal
        # Less dark   → rgb_w ≈ 0.48 → 52% thermal
        # Simple Fusion ALWAYS uses rgb_w = 0.50 — no adaptation
        rgb_w_log.append(rgb_w)

        fused     = rgb_w * rgb_f + (1.0 - rgb_w) * therm_f
        fused_bgr = (np.clip(fused, 0, 1) * 255).astype(np.uint8)
        cv2.imwrite(str(out_img_dir / f"{stem}.jpg"), fused_bgr)

        lbl_src = Path(lbl_src_dir) / (stem + ".txt")
        lbl_dst = out_lbl_dir / (stem + ".txt")
        if lbl_src.exists():
            shutil.copy(str(lbl_src), str(lbl_dst))
        else:
            lbl_dst.touch()

    total = len(list(out_img_dir.glob("*.jpg")))
    print(f"  {split}: {total:,} fused images | skipped: {skipped}")
    print(f"  RGB weight → min:{min(rgb_w_log):.4f}  max:{max(rgb_w_log):.4f}  "
          f"mean:{np.mean(rgb_w_log):.4f}  std:{np.std(rgb_w_log):.4f}")
    print(f"  Simple Fusion comparison: fixed 0.5000 for every image")

generate_fused(rgb_train, THERM_TRAIN, LBL_TRAIN,
               FUSED_TRAIN_DIR, FUSED_LBL_TRAIN, "train")
generate_fused(rgb_val,   THERM_VAL,   LBL_VAL,
               FUSED_VAL_DIR,   FUSED_LBL_VAL,   "val")


Fusing train:   0%|          | 0/12025 [00:00<?, ?it/s]

  train: 12,025 fused images | skipped: 0
  RGB weight → min:0.3165  max:0.7676  mean:0.4263  std:0.0908
  Simple Fusion comparison: fixed 0.5000 for every image


Fusing val:   0%|          | 0/3463 [00:00<?, ?it/s]

  val: 3,463 fused images | skipped: 0
  RGB weight → min:0.3015  max:0.6977  mean:0.4094  std:0.0971
  Simple Fusion comparison: fixed 0.5000 for every image


In [11]:
# Cell 9: Write data.yaml for Ultralytics

data_yaml = WORK / "data.yaml"

data_config = {
    "path":  str(WORK / "fused_images"),
    "train": "images/train",
    "val":   "images/val",
    "nc":    1,
    "names": {0: "person"}
}

with open(data_yaml, "w") as f:
    yaml.dump(data_config, f)

# Verify it exists and print contents
print(f"data.yaml written to: {data_yaml}")
print(f"File exists: {data_yaml.exists()}")
print("\nContents:")
with open(data_yaml) as f:
    print(f.read())

# Verify image/label counts
print(f"Train images : {len(list(FUSED_TRAIN_DIR.glob('*.jpg'))):,}")
print(f"Val images   : {len(list(FUSED_VAL_DIR.glob('*.jpg'))):,}")
print(f"Train labels : {len(list(FUSED_LBL_TRAIN.glob('*.txt'))):,}")
print(f"Val labels   : {len(list(FUSED_LBL_VAL.glob('*.txt'))):,}")
print("\n Ready for training")


data.yaml written to: /kaggle/working/adaptive_fusion_v2/data.yaml
File exists: True

Contents:
names:
  0: person
nc: 1
path: /kaggle/working/adaptive_fusion_v2/fused_images
train: images/train
val: images/val

Train images : 12,025
Val images   : 3,463
Train labels : 12,025
Val labels   : 3,463

 Ready for training


In [12]:
# Cell 10: Native Ultralytics Training — CRASH PROOF VERSION

import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"]     = "disabled"

import ultralytics.utils.callbacks.wb as _wb
_wb.on_pretrain_routine_start = lambda trainer: None
_wb.on_train_epoch_end        = lambda trainer: None
_wb.on_train_end              = lambda trainer: None
_wb.callbacks                 = {}

import ultralytics.utils.callbacks.raytune as _raytune
_raytune.on_fit_epoch_end = lambda trainer: None
_raytune.callbacks        = {}

print("WandB patched ")
print("RayTune patched ")

train_model = YOLO("yolov8n.pt")

try:
    train_model.train(
        data=str(data_yaml),
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        optimizer="SGD",
        lr0=0.01,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        hsv_h=0.005,
        hsv_s=0.3,
        hsv_v=0.2,
        translate=0.1,
        scale=0.5,
        fliplr=0.5,
        mosaic=1.0,
        patience=20,
        workers=4,
        project="adaptive_fusion",
        name="v2_final",
        exist_ok=True,
        verbose=True,
    )
except Exception as e:
    # PyTorch 2.6 raises UnpicklingError in strip_optimizer AFTER training
    # This is a known bug — training and best.pt are NOT affected
    print(f"\n  Post-training cleanup error (safe to ignore): {type(e).__name__}")
    print("  Training weights already saved — continuing to results...")

print("\n Cell 10 complete — proceeding to Cell 11")


WandB patched 
RayTune patched 
New https://pypi.org/project/ultralytics/8.4.19 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/kaggle/working/adaptive_fusion_v2/data.yaml, epochs=100, time=None, patience=20, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=4, project=adaptive_fusion, name=v2_final, exist_ok=True, pretrained=True, optimizer=SGD, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes

100%|██████████| 755k/755k [00:00<00:00, 24.7MB/s]
2026-02-28 14:26:47.338582: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772288807.567955      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772288807.634727      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772288808.243911      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772288808.243946      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772288808.243949      2

Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/checks.py:640: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(True):


AMP: checks passed ✅


...

                   all       3463       8302      0.957      0.897       0.96      0.637



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     62/100      3.07G      1.206     0.5871      1.125         38        640: 100%|██████████| 752/752 [02:05<00:00,  6.00it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 109/109 [00:19<00:00,  5.61it/s]

                   all       3463       8302      0.954      0.894      0.959      0.636
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 42, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



62 epochs completed in 2.513 hours.

  Post-training cleanup error (safe to ignore): UnpicklingError
  Training weights already saved — continuing to results...

 Cell 10 complete — proceeding to Cell 11


In [13]:
# Cell 11: Extract Final Results
import pandas as pd
from pathlib import Path

RUN_DIR     = Path("/kaggle/working/adaptive_fusion/v2_final")
results_csv = RUN_DIR / "results.csv"

# Wait check — confirm file exists
if not results_csv.exists():
    print(" results.csv not found — training may not have finished yet")
    print(f"   Expected at: {results_csv}")
else:
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    print("Columns found:", list(df.columns))
    print(f"Total epochs trained: {len(df)}")

    best_idx   = df["metrics/mAP50(B)"].idxmax()
    best_row   = df.loc[best_idx]

    v2_map50   = float(best_row["metrics/mAP50(B)"])
    v2_map5095 = float(best_row["metrics/mAP50-95(B)"])
    v2_prec    = float(best_row["metrics/precision(B)"])
    v2_recall  = float(best_row["metrics/recall(B)"])
    best_ep    = int(best_row["epoch"])

    print("\n" + "=" * 55)
    print("ADAPTIVE FUSION V2 — FINAL RESULTS")
    print("=" * 55)
    print(f"  Best epoch      : {best_ep}")
    print(f"  mAP@0.5         : {v2_map50:.4f}")
    print(f"  mAP@0.5:0.95    : {v2_map5095:.4f}")
    print(f"  Precision       : {v2_prec:.4f}")
    print(f"  Recall          : {v2_recall:.4f}")
    print("=" * 55)
    print(f"\n  Simple Fusion   : mAP50=0.9621  mAP50-95=0.6357")
    print(f"  Thermal Baseline: mAP50=0.9658  mAP50-95=0.6607")
    print(f"\n  Beat Simple Fusion?   : {' YES' if v2_map50   > 0.9621 else ' No'}")
    print(f"  Beat Thermal Baseline?: {' YES' if v2_map50   > 0.9658 else ' No'}")

    # Save copy to WORK folder
    WORK = Path("/kaggle/working/adaptive_fusion_v2")
    df.to_csv(WORK / "adaptive_fusion_v2_results.csv", index=False)
    print(f"\nResults CSV saved ✓")


Columns found: ['epoch', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']
Total epochs trained: 62

ADAPTIVE FUSION V2 — FINAL RESULTS
  Best epoch      : 42
  mAP@0.5         : 0.9634
  mAP@0.5:0.95    : 0.6449
  Precision       : 0.9626
  Recall          : 0.9001

  Simple Fusion   : mAP50=0.9621  mAP50-95=0.6357
  Thermal Baseline: mAP50=0.9658  mAP50-95=0.6607

  Beat Simple Fusion?   :  YES
  Beat Thermal Baseline?:  No

Results CSV saved ✓


In [14]:
# Cell 12: Final Dissertation Results Table
import pandas as pd
from pathlib import Path

WORK   = Path("/kaggle/working/adaptive_fusion_v2")
RUN_DIR = Path("/kaggle/working/adaptive_fusion/v2_final")

# ── Read v2 results directly from results.csv (self-contained, no Cell 11 dependency) ──
results_csv = RUN_DIR / "results.csv"
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()
best_idx   = df["metrics/mAP50(B)"].idxmax()
best_row   = df.loc[best_idx]
v2_map50   = float(best_row["metrics/mAP50(B)"])
v2_map5095 = float(best_row["metrics/mAP50-95(B)"])
v2_prec    = float(best_row["metrics/precision(B)"])
v2_recall  = float(best_row["metrics/recall(B)"])
# ─────────────────────────────────────────────────────────────────────────────

final_table = [
    ["RGB Baseline",       0.8946, 0.5255, 0.9210, 0.8245],
    ["Simple Fusion",      0.9621, 0.6357, 0.9600, 0.9025],
    ["Thermal Baseline",   0.9658, 0.6607, 0.9644, 0.9056],
    ["Adaptive Fusion v1", 0.9596, 0.6174, 0.9521, 0.9000],
    ["Adaptive Fusion v2", v2_map50, v2_map5095, v2_prec, v2_recall],
]

results_df = pd.DataFrame(
    final_table,
    columns=["Method", "mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall"]
)
results_df.to_csv(WORK / "FINAL_RESULTS_TABLE.csv", index=False)

print("=" * 68)
print("DISSERTATION FINAL RESULTS TABLE")
print("=" * 68)
print(f"  {'Method':<22} | {'mAP@0.5':>8} | {'mAP50-95':>8} | {'Prec':>7} | {'Recall':>7}")
print("  " + "-" * 64)
for row in final_table:
    if row[0] == "Simple Fusion":
        marker = " ← BENCHMARK"
    elif row[0] == "Adaptive Fusion v2":
        marker = " ← YOUR RESULT"
    else:
        marker = ""
    print(f"  {row[0]:<22} | {row[1]:>8.4f} | {row[2]:>8.4f} | "
          f"{row[3]:>7.4f} | {row[4]:>7.4f}{marker}")

print("=" * 68)
print(f"\nFusion weight analysis:")
print(f"  Simple Fusion  — fixed RGB weight : 0.5000 (every image)")
print(f"  Adaptive v2    — RGB weight range : 0.3015 – 0.7676 (per image)")
print(f"  RGB weight std : 0.0908  (proves genuine per-image adaptivity)")
print(f"\nFINAL_RESULTS_TABLE.csv saved ✓")


DISSERTATION FINAL RESULTS TABLE
  Method                 |  mAP@0.5 | mAP50-95 |    Prec |  Recall
  ----------------------------------------------------------------
  RGB Baseline           |   0.8946 |   0.5255 |  0.9210 |  0.8245
  Simple Fusion          |   0.9621 |   0.6357 |  0.9600 |  0.9025 ← BENCHMARK
  Thermal Baseline       |   0.9658 |   0.6607 |  0.9644 |  0.9056
  Adaptive Fusion v1     |   0.9596 |   0.6174 |  0.9521 |  0.9000
  Adaptive Fusion v2     |   0.9634 |   0.6449 |  0.9626 |  0.9001 ← YOUR RESULT

Fusion weight analysis:
  Simple Fusion  — fixed RGB weight : 0.5000 (every image)
  Adaptive v2    — RGB weight range : 0.3015 – 0.7676 (per image)
  RGB weight std : 0.0908  (proves genuine per-image adaptivity)

FINAL_RESULTS_TABLE.csv saved ✓
